# ReviewIQ — Training Our Own Sentiment Model

Requested at the midterm: train a model instead of only using pretrained ones.

**Design:** features = the 384-dim sentence embeddings already computed for all five apps
(cached by the full-scale batch run — no re-embedding needed). Labels derived from star
ratings: 1–2★ = negative (0), 4–5★ = positive (1), 3★ dropped as ambiguous.

**Honest caveat stated up front (circularity):** these labels ARE the star ratings, so
evaluating against stars favors the trained model by construction. The pretrained
transformer never saw stars. That is why the decisive experiment is **leave-one-app-out**
(train on four apps, test on the unseen fifth) — it measures what actually matters for
the product: generalization to apps we have no data for.

Models trained: **logistic regression** (simplest credible learner) and
**gradient boosting** (HistGradientBoosting — same family as XGBoost).

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

BASE = Path("../data/processed/full_run")
APPS = ["chatgpt", "facebook", "netflix", "snapchat", "tiktok"]

# Load every app's cached embeddings + reviews; build labels from stars.
data = {}
for app in APPS:
    emb = np.load(BASE / f"{app}_embeddings.npy")
    df = pd.read_parquet(BASE / f"{app}_clustered.parquet")
    keep = (df["score"] != 3).values          # drop ambiguous 3-star reviews
    data[app] = {
        "X": emb[keep],
        "y": (df.loc[keep, "score"] >= 4).astype(int).values,   # 1 = positive
        # the pretrained transformer's verdict for the SAME rows (for comparison)
        "transformer": (df.loc[keep, "sentiment"] == "positive").astype(int).values,
    }
    print(f"{app:9} kept {len(data[app]['y']):>6} reviews | positive share {data[app]['y'].mean():.2f}")

chatgpt   kept  55906 reviews | positive share 0.89
facebook  kept  50256 reviews | positive share 0.62


netflix   kept  84006 reviews | positive share 0.45


snapchat  kept  69660 reviews | positive share 0.56


tiktok    kept  68070 reviews | positive share 0.64


## Experiment 1 — Netflix, classic train/test split

The standard supervised setup: hold out 20% of reviews the model never sees during
training, grade it on those. `stratify` keeps the positive/negative ratio identical in
both halves; `random_state=42` makes the split reproducible.

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

X, y = data["netflix"]["X"], data["netflix"]["y"]
idx = np.arange(len(y))
idx_tr, idx_te = train_test_split(idx, test_size=0.2, random_state=42, stratify=y)

clf = LogisticRegression(max_iter=2000, random_state=42)
clf.fit(X[idx_tr], y[idx_tr])          # <-- the training happens here

print("train accuracy:", round(accuracy_score(y[idx_tr], clf.predict(X[idx_tr])), 3))
print("test  accuracy:", round(accuracy_score(y[idx_te], clf.predict(X[idx_te])), 3))
print()
print(classification_report(y[idx_te], clf.predict(X[idx_te]), target_names=["neg", "pos"], digits=2))

train accuracy: 0.897
test  accuracy: 0.897

              precision    recall  f1-score   support

         neg       0.89      0.93      0.91      9316
         pos       0.91      0.86      0.88      7486

    accuracy                           0.90     16802
   macro avg       0.90      0.89      0.90     16802
weighted avg       0.90      0.90      0.90     16802



## Experiment 2 — Gradient boosting on the same split

HistGradientBoosting is scikit-learn's fast gradient-boosted-trees implementation —
the same model family as XGBoost. Same data, same split, so the comparison is fair.

In [3]:
from sklearn.ensemble import HistGradientBoostingClassifier

gb = HistGradientBoostingClassifier(random_state=42)
gb.fit(X[idx_tr], y[idx_tr])
print("gradient boosting test accuracy:", round(accuracy_score(y[idx_te], gb.predict(X[idx_te])), 3))

# And the pretrained transformer on the SAME held-out rows (it never saw any stars):
t = data["netflix"]["transformer"][idx_te]
print("pretrained transformer agreement:", round(accuracy_score(y[idx_te], t), 3))

gradient boosting test accuracy: 0.89
pretrained transformer agreement: 0.829


## Experiment 3 — The one that matters: leave-one-app-out

The product's promise is working on apps we have **no data for**. So: train on four
apps, test on the fifth the model has never seen, rotate through all five. If the
trained model's advantage shrinks (or flips) here, that is evidence the pretrained
model's generality is worth its price — measured, not assumed.

In [4]:
rows = []
for held_out in APPS:
    train_apps = [a for a in APPS if a != held_out]
    X_tr = np.vstack([data[a]["X"] for a in train_apps])
    y_tr = np.concatenate([data[a]["y"] for a in train_apps])
    X_te, y_te = data[held_out]["X"], data[held_out]["y"]

    m = LogisticRegression(max_iter=2000, random_state=42)
    m.fit(X_tr, y_tr)

    rows.append({
        "held-out app": held_out,
        "trained model acc": round(accuracy_score(y_te, m.predict(X_te)), 3),
        "pretrained transformer acc": round(accuracy_score(y_te, data[held_out]["transformer"]), 3),
        "n test": len(y_te),
    })
    print("done:", held_out)

results = pd.DataFrame(rows)
results["winner"] = np.where(results["trained model acc"] > results["pretrained transformer acc"],
                             "trained", "pretrained")
results

done: chatgpt


done: facebook


done: netflix


done: snapchat


done: tiktok


,held-out app,trained model acc,pretrained transformer acc,n test,winner
0,chatgpt,0.932,0.899,55906,trained
1,facebook,0.859,0.825,50256,trained
2,netflix,0.875,0.827,84006,trained
3,snapchat,0.867,0.827,69660,trained
4,tiktok,0.832,0.796,68070,trained


## How to read the results

- **Experiments 1–2** show we can train competent models in minutes on top of the
  embedding representation — the embeddings do the heavy lifting; the classifier
  only learns a decision boundary in meaning-space.
- **The trained models' scores are inflated by construction**: they optimize star
  agreement, and star agreement is also the yardstick. The transformer never saw a
  star. Where text disagrees with stars ("5★ but please fix subtitles"), the trained
  model is *rewarded* for reproducing the star — the transformer for reading the text.
- **Experiment 3 is the fair fight**, and even it still uses stars as the yardstick —
  the fully honest evaluation would be a hand-labeled test set (planned: ~300
  disagreement cases labeled by us). What Experiment 3 does show is how much of the
  trained model's edge survives on an app it has never seen.
- **Product conclusion:** training is cheap and effective *when you have the target
  app's data*; the pretrained model is what makes the zero-setup, any-app promise
  possible. The two are complementary, not rivals — a deployed ReviewIQ could ship
  pretrained and fine-tune per customer over time.